# 02 — Topic Modeling (LDA)
**Proyek: Agentic Campus — Klasifikasi Topik Skripsi Teknik Informatika**  
**Mata Kuliah: Natural Language Processing**

Notebook ini mengeksplorasi pola topik dalam korpus skripsi menggunakan **Latent Dirichlet Allocation (LDA)**. Idenya: karena sebagian besar dokumen tidak berlabel, kita bisa melihat apakah LDA secara otomatis menemukan pengelompokan tematik yang masuk akal — misalnya topik yang berkaitan dengan keamanan jaringan, computer vision, atau NLP.

Hasil topic modeling ini bersifat eksploratif dan tidak dipakai langsung untuk klasifikasi; namun berguna untuk memvalidasi bahwa struktur topik yang kita asumsikan (4 kelas) memang ada di dalam data.


In [13]:
import os
import pandas as pd

# CountVectorizer -> Document-Term Matrix, LatentDirichletAllocation -> model LDA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

pd.set_option("display.max_colwidth", 200)

## 1. Load Dataset Bersih
Menggunakan hasil preprocessing dari notebook 01 — kolom `teks_bersih` yang sudah melalui normalisasi, tokenisasi, dan stopword removal.


In [14]:
CLEAN_PATH = "../data/processed/data_clean.csv"
df = pd.read_csv(CLEAN_PATH)

# Pastikan kolom teks tidak kosong / NaN sebelum vektorisasi
df["teks_bersih"] = df["teks_bersih"].fillna("").astype(str)
df = df[df["teks_bersih"].str.strip() != ""].reset_index(drop=True)

print(f"Jumlah dokumen: {len(df)}")
df[["teks_bersih"]].head(2)

Jumlah dokumen: 1136


,teks_bersih
0,klasifikasi kecenderungan depresi mahasiswa learning vector quantization depresi kondisi kesehatan serius dialami mahasiswa disadari depresi mengurangi depresi mencegah mahasiswa mengalami depresi...
1,secure chatting algoritma aes blockchain berbasis mobile mengusulkan pengamanan pesan kombinasi algoritma aes blockchain mengamankan pesan teks aplikasi chatting algoritma aes algoritma simetri me...


## 2. Document-Term Matrix (DTM)
Sebelum LDA bisa dilatih, teks perlu diubah ke representasi numerik. Kita pakai `CountVectorizer` (bukan TF-IDF, karena LDA bekerja dengan frekuensi mentah). Parameter yang dipakai:
- `max_df=0.95` — Buang kata yang muncul di lebih dari 95% dokumen (terlalu umum, tidak informatif untuk topik).
- `min_df=2` — Buang kata yang hanya muncul di satu dokumen (terlalu jarang, cenderung noise).
- `max_features=1000` — Batasi ke 1.000 term paling sering untuk menjaga efisiensi.


In [15]:
vectorizer = CountVectorizer(
    max_df=0.95,        # abaikan term yang muncul di > 95% dokumen (terlalu umum)
    min_df=2,           # abaikan term yang muncul di < 2 dokumen (terlalu jarang)
    max_features=1000,  # batasi ke 1000 term paling sering
)

dtm = vectorizer.fit_transform(df["teks_bersih"])
feature_names = vectorizer.get_feature_names_out()

print(f"Ukuran DTM: {dtm.shape[0]} dokumen x {dtm.shape[1]} term")

Ukuran DTM: 1136 dokumen x 1000 term


## 3. Latih Model LDA
Kita latih LDA dengan 6 topik. Jumlah ini sedikit lebih banyak dari 4 kelas yang ada, dengan harapan LDA bisa menemukan sub-topik yang lebih spesifik. `random_state` dikunci agar hasilnya reproducible setiap kali notebook dijalankan ulang.


In [16]:
N_TOPICS = 6

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=10,
    learning_method="batch",
    random_state=42,
)
lda.fit(dtm)

print(f"Model LDA dilatih dengan {N_TOPICS} topik.")
print(f"Perplexity: {lda.perplexity(dtm):.2f}")

Model LDA dilatih dengan 6 topik.
Perplexity: 417.97


## 4. Top 10 Keyword per Topik
Dari setiap topik, kita ambil 10 kata dengan bobot tertinggi. Dengan melihat kata-kata ini, kita bisa secara manual menamai topik tersebut dan mengevaluasi apakah LDA berhasil memisahkan tema dengan baik.


In [17]:
TOP_N = 10

def tampilkan_topik(model, feature_names, n_top_words=TOP_N):
    """Cetak n kata kunci teratas untuk setiap topik."""
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        keywords = [feature_names[i] for i in top_indices]
        print(f"Topik {topic_idx + 1}: {', '.join(keywords)}")

print("=== Top 10 Keyword per Topik ===")
tampilkan_topik(lda, feature_names)

=== Top 10 Keyword per Topik ===
Topik 1: algoritma, citra, rata, gambar, wajah, pengenalan, pesan, pengujian, game, waktu
Topik 2: model, akurasi, klasifikasi, neural, network, cnn, learning, convolutional, dataset, arsitektur
Topik 3: penyakit, pakar, keputusan, akurasi, pengujian, kriteria, pendukung, siswa, membantu, kulit
Topik 4: teks, algoritma, indonesia, means, informasi, clustering, bahasa, pengguna, model, lstm
Topik 5: fuzzy, prediksi, time, algoritma, series, harga, error, memprediksi, mape, backpropagation
Topik 6: klasifikasi, akurasi, algoritma, fitur, vector, rata, bayes, machine, support, svm


## 5. (Bonus) Topik Dominan per Dokumen & Simpan
Setiap dokumen diberi label topik berdasarkan probabilitas tertinggi dari distribusi topik LDA. Hasilnya disimpan ke `data/outputs/df_with_topics.csv` sebagai artefak eksplorasi — ini tidak dipakai dalam pipeline klasifikasi utama, tapi berguna untuk analisis lebih lanjut.


In [18]:
# Probabilitas tiap dokumen terhadap tiap topik
doc_topic = lda.transform(dtm)

# Topik dominan = indeks probabilitas terbesar (+1 agar penomoran mulai dari 1)
df["topik_dominan"] = doc_topic.argmax(axis=1) + 1
df["probabilitas"] = doc_topic.max(axis=1).round(4)

# Simpan hasil akhir
OUTPUT_PATH = "../data/outputs/df_with_topics.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Disimpan ke: {OUTPUT_PATH}")

# Distribusi dokumen per topik
print("\n=== Sebaran Dokumen per Topik ===")
print(df["topik_dominan"].value_counts().sort_index())
df[["teks_bersih", "topik_dominan", "probabilitas"]].head(5)

Disimpan ke: ../data/outputs/df_with_topics.csv

=== Sebaran Dokumen per Topik ===
topik_dominan
1    218
2    141
3    238
4    228
5    134
6    177
Name: count, dtype: int64


,teks_bersih,topik_dominan,probabilitas
0,klasifikasi kecenderungan depresi mahasiswa learning vector quantization depresi kondisi kesehatan serius dialami mahasiswa disadari depresi mengurangi depresi mencegah mahasiswa mengalami depresi...,6,0.9244
1,secure chatting algoritma aes blockchain berbasis mobile mengusulkan pengamanan pesan kombinasi algoritma aes blockchain mengamankan pesan teks aplikasi chatting algoritma aes algoritma simetri me...,1,0.9898
2,pendukung keputusan pemilihan produk whey protein susu gym terbaik smarter menunjang nutrisi tubuh gym pemula whey protein dianggap suplement membantu memenuhi nutrisi banyanya produk whey protein...,3,0.9542
3,komparasi moora waspas rekomendasi penetapan program mipa ips siswa sma sekolah penggerak tahun pelajaran perkembangan kebijakan program sekolah penggerak program sekolah penggerak diadopsi mengal...,3,0.9848
4,perbandingan dempster shafer na ve bayes pakar identifikasi jenis kulit wajah kulit wajah tubuh sensitif dijaga dirawat merawat kulit wajah jenis kulit wajah terhindar kulit iritasi jerawat merada...,3,0.8979
